# Recovering the characteristic function of a general linear combination of coordinate signatures from PCF

Let

$$ 
Z=\sum_{j=1}^{r}c_j\pi_{I_j}(S(X)),
$$


where $I_1,\ldots,I_r$ are arbitrary nonempty words. The aim is to recover
$$
    \phi_Z(\lambda)=\mathbb E\left[e^{i\lambda Z}\right]=\sum_{m=0}^{\infty}\frac{(i\lambda)^m}{m!}\mathbb E[Z^m]
$$

from the path characteristic function $\Phi_X$.

In [1]:
import numpy as np
import sympy as sp
from itertools import product
from math import factorial

## 1. The word-picker \(M_I\)

For a word $I=(i_1,\ldots,i_k)$, define $B_r=E_{r-1,r}-E_{r,r-1}$ for $r=1,\ldots,k,$ and $M_I(e_a)=\sum_{\substack{1\leq r\leq k\\ i_r=a}}B_r$ for $a=1,\ldots,d.$

The function `M_I(I, d)` returns the list $[M_I(e_1),\ldots,M_I(e_d)].$

In [4]:
def matrix_unit(N, p, q, dtype=float):
    # Return the unit matrix E_{p,q}.
    E = np.zeros((N, N), dtype=dtype)
    E[p][q] = 1
    return E


def matrix_B(k, r, dtype=float):
    # Return B_r = E_{r-1,r} - E_{r,r-1}, acting on C^{k+1}.
    if not (1 <= r <= k):
        raise ValueError("r must satisfy 1 <= r <= k.")

    return (matrix_unit(k + 1, r - 1, r, dtype) - matrix_unit(k + 1, r, r - 1, dtype))


def M_I(I, d, dtype=float):
    # Return [M_I(e_1), ..., M_I(e_d)].

    k = len(I)
    matrix_dim = k + 1

    MI = [np.zeros((matrix_dim, matrix_dim), dtype=dtype) for i in range(d)]

    for r, i in enumerate(I, start=1):
        if not (1 <= i <= d):
            raise ValueError("Every letter of I must lie in {1,...,d}.")

        MI[i - 1] += matrix_B(k, r, dtype)

    return MI

In [5]:
#check:
I = [1, 2]
d = 2
M12 = M_I(I, d)
print("M_12(e_1) =", M12[0])
print("\nM_12(e_2) =", M12[1])


M_12(e_1) = [[ 0.  1.  0.]
 [-1.  0.  0.]
 [ 0.  0.  0.]]

M_12(e_2) = [[ 0.  0.  0.]
 [ 0.  0.  1.]
 [ 0. -1.  0.]]


## 2. The tensor lift $M_I^{[m]}$ 
For one fixed word $I$, define:
$$M_I^{[m]}(e_a)
=
\sum_{q=1}^{m}
\operatorname{Id}^{\otimes(q-1)}
\otimes M_I(e_a)
\otimes
\operatorname{Id}^{\otimes(m-q)}.$$

In [6]:
def mat_tensor_prod(mats):
    # Return the Kronecker product of all matrices in mats.

    ans = np.array([[1.0]], dtype=mats[0].dtype)

    for M in mats:
        ans = np.kron(ans, M)

    return ans


def tensor_lift_word_picker(I, m, d, dtype=float):
    # Return [M_I^{[m]}(e_1), ..., M_I^{[m]}(e_d)].

    MI = M_I(I, d, dtype)

    k = len(I)
    matrix_dimension = k + 1
    Id = np.eye(matrix_dimension, dtype=dtype)

    lifted_dimension = matrix_dimension ** m
    MI_m = []

    for M in MI:
        lifted_M = np.zeros(
            (lifted_dimension, lifted_dimension),
            dtype=dtype
        )

        for q in range(m):
            mats = [Id for _ in range(m)]
            mats[q] = M

            lifted_M += mat_tensor_prod(mats)

        MI_m.append(lifted_M)

    return MI_m

## 3. Moment recovery from PCF

The PCF inversion formula gives
$$
    \mu_m(I)=\mathbb E[\pi_I(S(X))^m]=\frac{1}{(km)!}\left.\frac{d^{km}}{dt^{km}}[\Phi_X(tM_I^{[m]})]_{\mathbf 0,\mathbf k}\right|_{t=0}.
$$
 

In [ ]:
def pcf_dev(Phi_X, M, dev_ord, row_ind, col_ind):
    # Compute d^n/dt^n [Phi_X(tM)]_{row_index, column_index} at t = 0.

    t = sp.Symbol("t", real=True)

    # Represent tM by scaled_M: [tM(e_1), ..., tM(e_d)].
    scaled_M = []

    for M_a in M:
        scaled_M.append( t * sp.Matrix(M_a) )

    # Evaluate the PCF at tM:
    Phi_t = sp.Matrix( Phi_X(scaled_M) )

    f = Phi_t[row_ind, col_ind]

    # Differentiate with respect to t.
    dev = sp.diff(f,t,dev_ord)

    # Evaluate the derivative at t=0.
    dev_zero = dev.subs( t, 0)

    return sp.simplify(dev_zero)

In [8]:
def m_moment(I, m, d, Phi_X, dtype=float):
    # Recover mu_m(I) = E[pi_I(S(X))^m] from the PCF Phi_X.

    if m == 0:
        return 1

    k = len(I)

    # Construct M_I^[m].
    MI_m = tensor_lift_word_picker(I, m, d, dtype)

    dev_ord = k*m

    lifted_dim = (k + 1) ** m

    # bold 0 = (0,...,0) and k = (k,...,k):
    row_ind = 0
    col_ind = lifted_dim - 1

    # Compute d^{km}/dt^{km}[Phi_X(t M_I^[m])]_{bold 0,bold k} at t=0:
    derivative = pcf_dev(Phi_X, MI_m, dev_ord, row_ind, col_ind)

    # Taylor coefficient = derivative / (k*m)!.
    moment = derivative / factorial(dev_ord)

    return sp.simplify(moment)

In [9]:
def truncated_cf(I, lambda_val, trunc_ord, d, Phi_X, dtype=float):
    # Return \sum_{m=0}^{trunc_ord} (i*lambda_val)^m / m! * mu_m(I).
    # This is a finite truncation of the exact characteristic-function series.

    partial_sum = sp.Integer(0)
    #moments = []

    for m in range(trunc_ord + 1):
        moment = m_moment(I, m, d, Phi_X, dtype)
        #moments.append(moment)

        term = ((sp.I * lambda_val) ** m / factorial(m) * moment)

        partial_sum += term

    return sp.simplify(partial_sum)

## 5. Mixed tensor lift for different words

For words $I_1,\ldots,I_m$ ,define

$$
M_{I_1,\ldots,I_m}(e_a)
=
\sum_{q=1}^m
\operatorname{Id}\otimes\cdots\otimes
M_{I_q}(e_a)
\otimes\cdots\otimes\operatorname{Id}.
$$

Then

$$
\operatorname{Dev}_{tM_{I_1,\ldots,I_m}}(x)
=
\bigotimes_{q=1}^m
\operatorname{Dev}_{tM_{I_q}}(x).
$$

This allows the recovery of mixed moments involving arbitrary
coordinate words.

In [10]:
def mixed_tensor_lift( words, d, dtype=float):
    '''
        Construct the mixed tensor lift for words = [I_1, ..., I_m].
        Return [M_{I_1,...,I_m}(e_1), ..., M_{I_1,...,I_m}(e_d)].
    '''

    # Construct M_{I_q} for every word I_j:
    word_pickers = []

    for I in words:
        word_pickers.append(M_I(I, d, dtype))

    # Identity matrix for every tensor component.
    identities = []

    for I in words:
        identities.append( np.eye(len(I) + 1, dtype=dtype) )

    # Product of all component dimensions.
    lifted_dimension = 1

    for I in words:
        lifted_dimension *= len(I) + 1

    mixed_M = []

    # Construct M_{I_1,...,I_m}(e_a) for a=1,...,d.
    for a in range(d):
        lifted_M_a = np.zeros((lifted_dimension, lifted_dimension), dtype=dtype)

        # Place M_{I_q}(e_a) in the q-th tensor component.
        for q in range(len(words)):
            matrices = []

            for j in range(len(words)):
                if j == q:
                    matrices.append(
                        word_pickers[j][a]
                    )
                else:
                    matrices.append(
                        identities[j]
                    )

            lifted_M_a += mat_tensor_prod(
                matrices
            )

        mixed_M.append(lifted_M_a)

    return mixed_M

In [11]:
words = [[1, 2], [1]]
d = 2
mixed_M = mixed_tensor_lift(words, d)

M12 = M_I([1, 2], d)
M1 = M_I([1], d)

I3 = np.eye(3)
I2 = np.eye(2)

print(mixed_M[0] == mat_tensor_prod([M12[0], I2]) + mat_tensor_prod([I3, M1[0]]))
print(mixed_M[1] == mat_tensor_prod([M12[1], I2]) + mat_tensor_prod([I3, M1[1]]))


[[ True  True  True  True  True  True]
 [ True  True  True  True  True  True]
 [ True  True  True  True  True  True]
 [ True  True  True  True  True  True]
 [ True  True  True  True  True  True]
 [ True  True  True  True  True  True]]
[[ True  True  True  True  True  True]
 [ True  True  True  True  True  True]
 [ True  True  True  True  True  True]
 [ True  True  True  True  True  True]
 [ True  True  True  True  True  True]
 [ True  True  True  True  True  True]]


## 6. Recovery of a mixed moment

For `words = [I_1, ..., I_m]`, the function below constructs the mixed lift, sets
$$
K=|I_1|+\cdots+|I_m|,\qquad D=\prod_{q=1}^{m}(|I_q|+1),
$$
extracts entry `(0, D-1)` of $\Phi_X(tM_{I_1,\ldots,I_m})$, i.e. the $K$-th derivative at zero divided by $K!$.

In [12]:
def mixed_moment(words, d, Phi_X, dtype=float):
    '''
        Compute the mixed moment E[pi_{I_1}(S(X)) ... pi_{I_m}(S(X))] with words = [I_1, ..., I_m].
    '''

    if len(words) == 0:
        return 1

    # Construct the mixed tensor lift.
    mixed_M = mixed_tensor_lift(words, d, dtype)

    # derivative order: |I_1| + ... + |I_m|.
    derivative_order = 0
    for I in words:
        derivative_order += len(I)

    # Dimension of the mixed tensor space.
    lifted_dimension = 1

    for I in words:
        lifted_dimension *= len(I) + 1

    row_index = 0
    column_index = lifted_dimension - 1

    derivative = pcf_dev(Phi_X, mixed_M, derivative_order, row_index, column_index)

    moment = derivative / factorial(derivative_order)

    return sp.simplify(moment)

## 7. Moments of a linear combination

Let 
$$ Z = \sum_{j=1}^{r} c_j \pi_{I_j}(S(X)) $$
then
$$ Z^m = \sum_{j_1,\dots,j_m=1}^{r} c_{j_1} \dots c_{j_m} \pi_{I_{j_1}}(S(X)) \dots \pi_{I_{j_m}}(S(X)) $$

In [13]:
def linear_combination_moment(linear_comb, m, d, Phi_X, dtype=float):
    '''
        Recover E[Z^m] for Z = sum_j c_j pi_{I_j}(S(X)).
        Each item in linear_comb is a tuple (coefficient, word).
    '''
    if m == 0:
        return 1

    term_num = len(linear_comb)

    total_moment = sp.Integer(0)

    # Generate all ordered tuples (j_1,...,j_m).
    for i in product(range(term_num), repeat=m):
        coeff_product = sp.Integer(1)
        words = []

        for j in i:
            coeff = linear_comb[j][0]
            word = linear_comb[j][1]

            coeff_product *= sp.sympify(coeff)
            words.append(word)

        current_mixed_moment = mixed_moment(words, d, Phi_X, dtype)

        total_moment += coeff_product * current_mixed_moment

    return sp.simplify(total_moment)

In [14]:
def linear_comb_truncated_cf(linear_comb, lambda_val, trunc_ord, d, Phi_X, dtype=float):
    '''
        Return the finite truncation \sum_{m=0}^{trunc_ord}(i*lambda_val)^m / m! * E[Z^m],
        where Z = \sum_{j} c_j pi_{I_j}(S(X)) and each item in linear_comb is a tuple (coefficient, word).
    '''
    partial_sum = sp.Integer(0)
    moments = []

    for m in range(trunc_ord + 1):
        moment = linear_combination_moment(linear_comb, m, d, Phi_X, dtype)
        moments.append(moment)

        term = ((sp.I * lambda_val) ** m / factorial(m) * moment)

        partial_sum += term

    return sp.simplify(partial_sum), moments

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
C:\Users\happy\AppData\Local\Temp\ipykernel_15660\3939255402.py:2: SyntaxWarning: invalid escape sequence '\s'
  '''


## 8. The remainder bound

If the driving path has total $1$-variation bounded by $L$ almost surely, then
$$
|Z|\leq \sum_{j=1}^{r}|c_j|\frac{L^{|I_j|}}{|I_j|!}.
$$
The next function converts this deterministic bound into an absolute Taylor-tail bound. It is not applicable to Brownian sample paths, because Brownian motion has infinite $1$-variation almost surely.

In [15]:
def remainder_bound(linear_comb, lambda_val, trunc_ord, L):
    '''
        Return a bound for the remainder of the truncated characteristic-function series
        \sum_{m=0}^{trunc_ord}(i*lambda_val)^m / m! * E[Z^m].
    '''
    L = sp.sympify(L)
    Z_bd = 0

    for coefficient, word in linear_comb:
        k = len(word)

        Z_bd += sp.Abs(sp.sympify(coefficient)) * L ** k / sp.factorial(k)

    m = sp.Symbol( "m", integer=True, nonnegative=True)

    return sp.Sum((sp.Abs(lambda_val) * Z_bd) ** m / sp.factorial(m), (m, trunc_ord + 1, sp.oo))

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
C:\Users\happy\AppData\Local\Temp\ipykernel_15660\2276461866.py:2: SyntaxWarning: invalid escape sequence '\s'
  '''


###  The Levy area of Brownian motion:
For d-dimensional Brownian motion $ B_{[0,T]} $, 
$$\Phi_B(M) = \exp\left( \frac{T}{2} \sum_{a=1}^{d} M(e_a)^2 \right)$$.


In [16]:
def Phi_B(M, T=1):
    M = [sp.Matrix(M_a) for M_a in M]

    R = sp.zeros(M[0].rows, M[0].cols)
    for M_a in M:
        R += M_a ** 2

    return (sp.sympify(T) * R / 2).exp()

In [20]:
lam = sp.symbols("lambda", real=True)

linear_comb = [
    (1, [1, 2]),
    (-1, [2, 1])
]

phi_Z_ord4, recovered_moments = linear_comb_truncated_cf(
    linear_comb,
    lambda_val=lam,
    trunc_ord=2,
    d=2,
    Phi_X=Phi_B,
    dtype=int
)

print(recovered_moments,"\n", phi_Z_ord2)


KeyboardInterrupt: 

## 8. Brownian validation: the antisymmetric level-two coordinate

For standard $d$-dimensional Brownian motion $B$ on $[0,T]$, interpreted with its Stratonovich lift,
$$
\Phi_B(M) = \exp\left( \frac{T}{2} \sum_{a=1}^{d} M(e_a)^2 \right).
$$
We test the general linear-combination algorithm on
$$
Z_T=\pi_{12}(S(B_{[0,T]}))-\pi_{21}(S(B_{[0,T]})).
$$
With this convention, $\phi_{Z_T}(\lambda)=\operatorname{sech}(\lambda T)$.  

Brownian scaling gives $B_{[0,T]}\overset{d}{=}\sqrt{T}B_{[0,1]}$ after time rescaling. Since both coordinates above have level two, $Z_T\overset{d}{=}T Z_1$. Therefore it is enough to validate the code at $T=1$; the general-$T$ moments follow by $\mathbb E[Z_T^m]=T^m\mathbb E[Z_1^m]$.